In [1]:
!pip install yfinance plotly dash jupyter-dash --quiet
print("✅ Libraries installed")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 57.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 80.2 MB/s eta 0:00:00
✅ Libraries installed


In [2]:
import yfinance as yf
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

tickers = ["JPM", "GS", "BAC", "MS", "C"]
nombres = {
    "JPM": "JPMorgan Chase",
    "GS": "Goldman Sachs",
    "BAC": "Bank of America",
    "MS": "Morgan Stanley",
    "C": "Citigroup"
}

datos = yf.download(tickers, period="1y", auto_adjust=True)['Close']
retornos = (datos / datos.iloc[0] - 1) * 100
volatilidad = datos.pct_change().rolling(30).std() * 100

print("✅ Data loaded")
print(f"Period: {datos.index[0].date()} → {datos.index[-1].date()}")

[*********************100%***********************]  5 of 5 completed

✅ Data loaded
Period: 2025-05-06 → 2026-05-05


In [3]:
kpis = pd.DataFrame({
    'Company': [nombres[t] for t in tickers],
    'Current Price': [round(datos[t].iloc[-1], 2) for t in tickers],
    'Return 1Y (%)': [round(retornos[t].iloc[-1], 1) for t in tickers],
    'Volatility (%)': [round(datos[t].pct_change().std() * 100, 2) for t in tickers],
    'Max Price': [round(datos[t].max(), 2) for t in tickers],
    'Min Price': [round(datos[t].min(), 2) for t in tickers],
}).sort_values('Return 1Y (%)', ascending=False).reset_index(drop=True)

print("📊 KPI SUMMARY")
print(kpis.to_string(index=False))

📊 KPI SUMMARY
        Company  Current Price  Return 1Y (%)  Volatility (%)  Max Price  Min Price
      Citigroup         128.01           88.4            1.76     132.42      67.93
  Goldman Sachs         918.89           70.7            1.70     970.75     538.21
 Morgan Stanley         189.25           63.8            1.59     190.60     115.54
Bank of America          53.12           33.0            1.36      56.93      39.94
 JPMorgan Chase         309.40           26.5            1.33     332.91     244.50


In [4]:
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        "Stock Price — Last 12 Months",
        "Comparative Return (%)",
        "30-Day Rolling Volatility (%)",
        "Annual Return by Company (%)"
    ),
    vertical_spacing=0.12,
    horizontal_spacing=0.08
)

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']

for i, ticker in enumerate(tickers):
    # Panel 1: Price
    fig.add_trace(go.Scatter(
        x=datos.index, y=datos[ticker],
        name=nombres[ticker], line=dict(color=colors[i], width=2),
        showlegend=True
    ), row=1, col=1)

    # Panel 2: Comparative return
    fig.add_trace(go.Scatter(
        x=retornos.index, y=retornos[ticker],
        name=nombres[ticker], line=dict(color=colors[i], width=2),
        showlegend=False
    ), row=1, col=2)

    # Panel 3: Volatility
    fig.add_trace(go.Scatter(
        x=volatilidad.index, y=volatilidad[ticker],
        name=nombres[ticker], line=dict(color=colors[i], width=2),
        showlegend=False
    ), row=2, col=1)

# Panel 4: Bar chart of annual return
fig.add_trace(go.Bar(
    x=kpis['Company'],
    y=kpis['Return 1Y (%)'],
    marker_color=colors[:len(tickers)],
    showlegend=False,
    text=kpis['Return 1Y (%)'].astype(str) + '%',
    textposition='outside'
), row=2, col=2)

fig.update_layout(
    title=dict(
        text="Financial Dashboard — Major US Banks (Last 12 Months)",
        font=dict(size=20, color='#1a1a1a'),
        x=0.5
    ),
    height=700,
    template='plotly_white',
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1)
)

fig.write_html("financial_dashboard.html")
fig.show()
print("✅ Dashboard saved as financial_dashboard.html")

✅ Dashboard saved as financial_dashboard.html


In [5]:
fig2 = go.Figure(data=[go.Table(
    header=dict(
        values=['<b>Company</b>', '<b>Price (USD)</b>', '<b>Return 1Y (%)</b>',
                '<b>Volatility (%)</b>', '<b>Max</b>', '<b>Min</b>'],
        fill_color='#1f4e79',
        font=dict(color='white', size=13),
        align='center',
        height=35
    ),
    cells=dict(
        values=[
            kpis['Company'],
            kpis['Current Price'],
            kpis['Return 1Y (%)'],
            kpis['Volatility (%)'],
            kpis['Max Price'],
            kpis['Min Price']
        ],
        fill_color=[['#f0f4f8' if i%2==0 else 'white'
                     for i in range(len(kpis))]],
        font=dict(size=12),
        align='center',
        height=30
    )
)])

fig2.update_layout(
    title=dict(text="KPI Summary — Major US Banks", font=dict(size=18), x=0.5),
    height=300
)

fig2.write_html("kpi_table.html")
fig2.show()
print("✅ KPI table saved")

✅ KPI table saved
